# Notebook 3: AnnData — Structure, Exploration, and Manipulation

**Objective:** Understand the AnnData object structure, create AnnData objects from scratch, explore all slots (`.X`, `.obs`, `.var`, `.uns`, `.obsm`, `.layers`), practice subsetting/slicing, and demonstrate concatenation.

**Reference Tutorials:**
- [AnnData Getting Started](https://anndata.readthedocs.io/en/latest/tutorials/notebooks/getting-started.html)
- [scverse — AnnData Getting Started](https://scverse-tutorials.readthedocs.io/en/latest/notebooks/anndata_getting_started.html)

## 1. Setup and Imports

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print(f"AnnData version: {ad.__version__}")

AnnData version: 0.11.4


## 2. Creating an AnnData Object from Scratch

An AnnData object can be constructed from a count matrix plus optional metadata. This is the foundation of the data structure used throughout single-cell analysis.

In [2]:
# Create a small synthetic count matrix: 5 cells x 4 genes
np.random.seed(42)
counts = np.random.poisson(lam=3, size=(5, 4)).astype(np.float32)

# Cell and gene identifiers
cell_ids = [f"Cell_{i}" for i in range(1, 6)]
gene_ids = ["GeneA", "GeneB", "GeneC", "GeneD"]

# Build AnnData object
adata_toy = ad.AnnData(
    X=counts,
    obs=pd.DataFrame(index=cell_ids),
    var=pd.DataFrame(index=gene_ids)
)

print(adata_toy)
print(f"\nExpression matrix (adata.X):")
print(adata_toy.X)

AnnData object with n_obs × n_vars = 5 × 4

Expression matrix (adata.X):
[[4. 1. 3. 3.]
 [2. 3. 2. 3.]
 [0. 2. 4. 2.]
 [1. 3. 4. 4.]
 [1. 2. 3. 2.]]


## 3. Understanding AnnData Slots

The AnnData object has several key slots:

| Slot | Shape | Description |
|------|-------|-------------|
| `.X` | (n_obs, n_vars) | The main data matrix (expression values) |
| `.obs` | (n_obs, ) | Cell-level metadata (DataFrame) |
| `.var` | (n_vars, ) | Gene-level metadata (DataFrame) |
| `.uns` | dict | Unstructured annotations (parameters, colors, etc.) |
| `.obsm` | dict of arrays | Multi-dimensional cell annotations (PCA, UMAP coords) |
| `.varm` | dict of arrays | Multi-dimensional gene annotations |
| `.layers` | dict of matrices | Alternative representations of X (raw counts, normalized) |
| `.obsp` | dict of sparse matrices | Cell-cell pairwise data (distances, connectivities) |

### 3.1 Adding Cell Metadata (`.obs`)

In [3]:
# Add metadata to cells
adata_toy.obs['cell_type'] = ['T cell', 'B cell', 'Monocyte', 'NK cell', 'T cell']
adata_toy.obs['n_counts'] = counts.sum(axis=1)
adata_toy.obs['condition'] = ['control', 'control', 'treated', 'treated', 'control']

print("Cell metadata (adata.obs):")
print(adata_toy.obs)

Cell metadata (adata.obs):
       cell_type  n_counts condition
Cell_1    T cell      11.0   control
Cell_2    B cell      10.0   control
Cell_3  Monocyte       8.0   treated
Cell_4   NK cell      12.0   treated
Cell_5    T cell       8.0   control


### 3.2 Adding Gene Metadata (`.var`)

In [4]:
# Add metadata to genes
adata_toy.var['highly_variable'] = [True, False, True, False]
adata_toy.var['gene_biotype'] = ['protein_coding', 'lncRNA', 'protein_coding', 'protein_coding']

print("Gene metadata (adata.var):")
print(adata_toy.var)

Gene metadata (adata.var):
       highly_variable    gene_biotype
GeneA             True  protein_coding
GeneB            False          lncRNA
GeneC             True  protein_coding
GeneD            False  protein_coding


### 3.3 Unstructured Annotations (`.uns`)

In [5]:
# Store unstructured metadata
adata_toy.uns['experiment'] = 'PBMC scRNA-seq demo'
adata_toy.uns['author'] = 'Asim Ahmed'
adata_toy.uns['preprocessing_params'] = {
    'min_genes': 200,
    'max_genes': 2500,
    'max_mt_pct': 5.0
}

print("Unstructured annotations (adata.uns):")
for key, val in adata_toy.uns.items():
    print(f"  {key}: {val}")

Unstructured annotations (adata.uns):
  experiment: PBMC scRNA-seq demo
  author: Asim Ahmed
  preprocessing_params: {'min_genes': 200, 'max_genes': 2500, 'max_mt_pct': 5.0}


### 3.4 Multi-dimensional Annotations (`.obsm`)

In [6]:
# Simulate PCA and UMAP coordinates
adata_toy.obsm['X_pca'] = np.random.randn(5, 3).astype(np.float32)   # 3 PCs
adata_toy.obsm['X_umap'] = np.random.randn(5, 2).astype(np.float32)  # 2D UMAP

print("Keys in obsm:", list(adata_toy.obsm.keys()))
print(f"\nPCA coordinates (X_pca):")
print(adata_toy.obsm['X_pca'])
print(f"\nUMAP coordinates (X_umap):")
print(adata_toy.obsm['X_umap'])

Keys in obsm: ['X_pca', 'X_umap']

PCA coordinates (X_pca):
[[ 0.5821228   0.8877485   0.89433235]
 [ 0.7549978  -0.2071659  -0.6234774 ]
 [-1.5081533   1.0996469  -0.17773212]
 [-0.4103833   1.1797163  -0.89820796]
 [ 0.8347954   0.2965614  -1.0378299 ]]

UMAP coordinates (X_umap):
[[-0.07580374  0.9729635 ]
 [ 0.79559547  1.4954343 ]
 [ 0.33818126  3.3722963 ]
 [-0.9203908  -0.3986384 ]
 [-0.06086409 -1.4187504 ]]


### 3.5 Layers — Alternative Data Representations

In [7]:
# Store raw counts in a layer, and a log-transformed version
adata_toy.layers['raw_counts'] = counts.copy()
adata_toy.layers['log_norm'] = np.log1p(counts / counts.sum(axis=1, keepdims=True) * 1e4)

print("Layers available:", list(adata_toy.layers.keys()))
print(f"\nRaw counts (layers['raw_counts']):")
print(adata_toy.layers['raw_counts'])
print(f"\nLog-normalized (layers['log_norm']):")
print(np.round(adata_toy.layers['log_norm'], 2))

Layers available: ['raw_counts', 'log_norm']

Raw counts (layers['raw_counts']):
[[4. 1. 3. 3.]
 [2. 3. 2. 3.]
 [0. 2. 4. 2.]
 [1. 3. 4. 4.]
 [1. 2. 3. 2.]]

Log-normalized (layers['log_norm']):
[[8.2  6.81 7.91 7.91]
 [7.6  8.01 7.6  8.01]
 [0.   7.82 8.52 7.82]
 [6.73 7.82 8.11 8.11]
 [7.13 7.82 8.23 7.82]]


## 4. Slicing and Subsetting

AnnData supports NumPy-style indexing. Slicing returns a **view** (not a copy), which is memory-efficient.

In [8]:
# Subset: first 3 cells, first 2 genes
subset = adata_toy[:3, :2]
print("Subset (3 cells, 2 genes):")
print(subset)
print(f"\nExpression values:")
print(subset.X)

Subset (3 cells, 2 genes):
View of AnnData object with n_obs × n_vars = 3 × 2
    obs: 'cell_type', 'n_counts', 'condition'
    var: 'highly_variable', 'gene_biotype'
    uns: 'experiment', 'author', 'preprocessing_params'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts', 'log_norm'

Expression values:
[[4. 1.]
 [2. 3.]
 [0. 2.]]


In [9]:
# Boolean indexing: select only T cells
t_cells = adata_toy[adata_toy.obs['cell_type'] == 'T cell']
print(f"T cells only: {t_cells.n_obs} cells")
print(t_cells.obs)

T cells only: 2 cells
       cell_type  n_counts condition
Cell_1    T cell      11.0   control
Cell_5    T cell       8.0   control


In [10]:
# Select specific genes by name
hvg_subset = adata_toy[:, adata_toy.var['highly_variable']]
print(f"HVG subset: {hvg_subset.n_obs} cells x {hvg_subset.n_vars} genes")
print(f"Gene names: {hvg_subset.var_names.tolist()}")

HVG subset: 5 cells x 2 genes
Gene names: ['GeneA', 'GeneC']


## 5. Concatenation

Multiple AnnData objects (e.g. different samples or batches) can be concatenated along the observation (cell) axis.

In [11]:
# Create a second small dataset (different "batch")
np.random.seed(99)
counts2 = np.random.poisson(lam=5, size=(3, 4)).astype(np.float32)
adata_batch2 = ad.AnnData(
    X=counts2,
    obs=pd.DataFrame({
        'cell_type': ['B cell', 'Monocyte', 'NK cell'],
        'condition': ['treated', 'treated', 'control']
    }, index=[f"Cell_{i}" for i in range(6, 9)]),
    var=pd.DataFrame(index=gene_ids)
)

# Concatenate
adata_combined = ad.concat(
    [adata_toy, adata_batch2],
    label='batch',
    keys=['batch1', 'batch2']
)
adata_combined.obs_names_make_unique()

print(f"Combined object: {adata_combined.n_obs} cells x {adata_combined.n_vars} genes")
print(f"\nBatch distribution:")
print(adata_combined.obs['batch'].value_counts())
print(f"\nAll cell metadata:")
print(adata_combined.obs)

Combined object: 8 cells x 4 genes

Batch distribution:
batch
batch1    5
batch2    3
Name: count, dtype: int64

All cell metadata:
       cell_type condition   batch
Cell_1    T cell   control  batch1
Cell_2    B cell   control  batch1
Cell_3  Monocyte   treated  batch1
Cell_4   NK cell   treated  batch1
Cell_5    T cell   control  batch1
Cell_6    B cell   treated  batch2
Cell_7  Monocyte   treated  batch2
Cell_8   NK cell   control  batch2


## 6. Sparse Matrices

Real scRNA-seq count matrices are very sparse (>90% zeros). AnnData supports scipy sparse matrices to save memory.

In [12]:
# Convert dense matrix to sparse
sparse_matrix = sp.csr_matrix(counts)
adata_sparse = ad.AnnData(X=sparse_matrix)

print(f"Dense matrix memory:  {counts.nbytes} bytes")
print(f"Sparse matrix type:   {type(adata_sparse.X)}")
print(f"Non-zero entries:     {adata_sparse.X.nnz} / {adata_sparse.X.shape[0] * adata_sparse.X.shape[1]}")
print(f"\nTo view sparse data, use .toarray():")
print(adata_sparse.X.toarray())

Dense matrix memory:  80 bytes
Sparse matrix type:   <class 'scipy.sparse._csr.csr_matrix'>
Non-zero entries:     19 / 20

To view sparse data, use .toarray():
[[4. 1. 3. 3.]
 [2. 3. 2. 3.]
 [0. 2. 4. 2.]
 [1. 3. 4. 4.]
 [1. 2. 3. 2.]]


## 7. Exploring Real Data — Clustered PBMC Dataset

Now we load the real clustered dataset from Notebook 2 and explore its AnnData structure.

In [13]:
# Load clustered data from Notebook 2 (relative path)
adata = ad.read_h5ad('../2_Basic_Tutorial/output_data/clustered_adata.h5ad')

print("=== Full AnnData Object ===")
print(adata)

=== Full AnnData Object ===
AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'cell_type'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'


### 7.1 Cell Metadata (`.obs`)

In [14]:
print("Cell metadata columns:", list(adata.obs.columns))
print(f"\nFirst 5 rows of .obs:")
display(adata.obs.head())

print(f"\nCluster distribution (leiden):")
print(adata.obs['leiden'].value_counts().sort_index())

Cell metadata columns: ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'cell_type']

First 5 rows of .obs:


,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,n_genes,leiden,cell_type
index,,,,,,,
AAACATACAACCAC-1,781,2421.0,73.0,3.015283,781,0,CD4 T cells
AAACATTGAGCTAC-1,1352,4903.0,186.0,3.793596,1352,3,CD8 T cells
AAACATTGATCAGC-1,1131,3149.0,28.0,0.889171,1131,0,CD4 T cells
AAACCGTGCTTCCG-1,960,2639.0,46.0,1.743085,960,1,CD14 Monocytes
AAACCGTGTATGCG-1,522,981.0,12.0,1.223242,522,2,B cells



Cluster distribution (leiden):
leiden
0    1182
1     636
2     430
3     341
4      36
5      13
Name: count, dtype: int64


### 7.2 Gene Metadata (`.var`)

In [15]:
print("Gene metadata columns:", list(adata.var.columns))
print(f"\nFirst 5 rows of .var:")
display(adata.var.head())

if 'highly_variable' in adata.var.columns:
    print(f"\nHighly variable genes: {adata.var['highly_variable'].sum()}")

Gene metadata columns: ['gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std']

First 5 rows of .var:


,gene_ids,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,highly_variable,means,dispersions,dispersions_norm,mean,std
index,,,,,,,,,,,,,
TNFRSF4,ENSG00000186827,False,155,0.077407,94.259259,209.0,155,True,0.277410,2.086050,0.665406,7.927981e-11,0.424483
CPSF3L,ENSG00000127054,False,202,0.094815,92.518519,256.0,202,True,0.385194,4.506987,2.955005,-3.163249e-10,0.460415
ATAD3C,ENSG00000215915,False,9,0.009259,99.666667,25.0,9,True,0.038252,3.953486,4.352607,5.189705e-11,0.119465
C1orf86,ENSG00000162585,False,501,0.227778,81.444444,615.0,501,True,0.678283,2.713522,0.543183,5.648659e-11,0.685147
RER1,ENSG00000157916,False,608,0.298148,77.481481,805.0,608,True,0.814813,3.447533,1.582528,-3.127945e-10,0.736054



Highly variable genes: 1838


### 7.3 Unstructured Annotations (`.uns`)

In [16]:
print("Keys in adata.uns:")
for key in adata.uns.keys():
    val = adata.uns[key]
    if isinstance(val, dict):
        print(f"  {key}: dict with keys {list(val.keys())[:5]}...")
    elif isinstance(val, np.ndarray):
        print(f"  {key}: array of shape {val.shape}")
    else:
        print(f"  {key}: {type(val).__name__}")

Keys in adata.uns:
  cell_type_colors: array of shape (6,)
  dendrogram_leiden: dict with keys ['categories_idx_ordered', 'categories_ordered', 'cor_method', 'correlation_matrix', 'dendrogram_info']...
  hvg: dict with keys ['flavor']...
  leiden: dict with keys ['params']...
  leiden_colors: array of shape (6,)
  log1p: dict with keys []...
  neighbors: dict with keys ['connectivities_key', 'distances_key', 'params']...
  pca: dict with keys ['params', 'variance', 'variance_ratio']...
  rank_genes_groups: dict with keys ['logfoldchanges', 'names', 'params', 'pvals', 'pvals_adj']...
  umap: dict with keys ['params']...


### 7.4 Multi-dimensional Embeddings (`.obsm`)

In [17]:
print("Keys in adata.obsm:", list(adata.obsm.keys()))
for key in adata.obsm.keys():
    print(f"  {key}: shape {adata.obsm[key].shape}")

# Preview UMAP coordinates
if 'X_umap' in adata.obsm:
    print(f"\nFirst 5 UMAP coordinates:")
    print(adata.obsm['X_umap'][:5])

Keys in adata.obsm: ['X_pca', 'X_umap']
  X_pca: shape (2638, 50)
  X_umap: shape (2638, 2)

First 5 UMAP coordinates:
[[-0.8914937  4.645378 ]
 [ 2.3830783 -2.2581644]
 [-4.5713787  3.2698824]
 [10.087383   9.631019 ]
 [-2.7934356  9.211127 ]]


### 7.5 Pairwise Annotations (`.obsp`)

In [18]:
print("Keys in adata.obsp:", list(adata.obsp.keys()))
for key in adata.obsp.keys():
    mat = adata.obsp[key]
    print(f"  {key}: shape {mat.shape}, non-zero entries: {mat.nnz}")

Keys in adata.obsp: ['connectivities', 'distances']
  connectivities: shape (2638, 2638), non-zero entries: 41966
  distances: shape (2638, 2638), non-zero entries: 23742


### 7.6 Accessing the Raw Data

In [19]:
if adata.raw is not None:
    print(f"Raw data available: {adata.raw.shape[0]} cells x {adata.raw.shape[1]} genes")
    print(f"(Main matrix has {adata.n_vars} genes — raw retains all genes for visualization)")
else:
    print("No raw data stored in this object.")

Raw data available: 2638 cells x 13714 genes
(Main matrix has 1838 genes — raw retains all genes for visualization)


## 8. Slicing the Real Dataset

In [20]:
# Slice: first 3 cells, first 5 genes
sliced = adata[:3, :5]
print(f"Sliced object: {sliced}")
print(f"\nExpression matrix (dense view):")
if sp.issparse(sliced.X):
    print(sliced.X.toarray())
else:
    print(sliced.X)

Sliced object: View of AnnData object with n_obs × n_vars = 3 × 5
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'leiden', 'cell_type'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Expression matrix (dense view):
[[-0.17157319 -0.28071612 -0.04664239 -0.47522914 -0.5440646 ]
 [-0.21422334 -0.37253138 -0.05476852 -0.68307877  0.6344757 ]
 [-0.37678805 -0.2952973  -0.05759941 -0.5208889   1.3326776 ]]


## 9. Exporting Data

Export cell metadata to CSV for use in other tools (e.g. R, Excel).

In [21]:
# Export cell metadata
adata.obs.to_csv('output_data/cell_metadata.csv')
print(f"Exported cell metadata: output_data/cell_metadata.csv")
print(f"Rows: {adata.obs.shape[0]}, Columns: {adata.obs.shape[1]}")

# Export UMAP coordinates
if 'X_umap' in adata.obsm:
    umap_df = pd.DataFrame(
        adata.obsm['X_umap'],
        index=adata.obs_names,
        columns=['UMAP1', 'UMAP2']
    )
    umap_df.to_csv('output_data/umap_coordinates.csv')
    print(f"Exported UMAP coordinates: output_data/umap_coordinates.csv")

Exported cell metadata: output_data/cell_metadata.csv
Rows: 2638, Columns: 7
Exported UMAP coordinates: output_data/umap_coordinates.csv


---
## Summary

| Concept | Description |
|---------|-------------|
| `AnnData` creation | `ad.AnnData(X=matrix, obs=cell_df, var=gene_df)` |
| `.X` | Main expression matrix (dense or sparse) |
| `.obs` / `.var` | Cell and gene metadata DataFrames |
| `.uns` | Unstructured annotations (dicts, params, colors) |
| `.obsm` | Multi-dimensional embeddings (PCA, UMAP) |
| `.layers` | Alternative data representations (raw, normalized) |
| `.obsp` | Pairwise cell metrics (distances, connectivities) |
| Slicing | NumPy-style: `adata[cells, genes]`, boolean indexing |
| Concatenation | `ad.concat([adata1, adata2], label='batch')` |
| I/O | `adata.write('file.h5ad')` / `ad.read_h5ad('file.h5ad')` |